In [5]:
# ============================================================
# Name: Sagar
# Roll No: 2301010312
# Course: Digital Image Processing
# Unit: Image Restoration
# Assignment Title: Noise Modeling and Restoration Analysis
# Date: 11 / 02 / 2026
# ============================================================

import cv2
import numpy as np
import os
import math

# ------------------ Welcome ------------------ #

print("\n==============================================")
print(" Image Restoration & Noise Modeling System ")
print(" Tasks:")
print(" 1. Load surveillance-style image")
print(" 2. Add Gaussian & Salt-Pepper noise")
print(" 3. Restore using spatial filters")
print(" 4. Compute MSE & PSNR")
print(" 5. Print analytical comparison")
print("==============================================\n")

# Output directory
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------ Utilities ------------------ #

def load_image():
    path = input("Enter surveillance image path: ")
    img = cv2.imread(path)

    if img is None:
        print("Error loading image.")
        exit()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    cv2.imwrite(f"{OUTPUT_DIR}/original.png", gray)
    print("Original image saved.")

    return gray


# -------- Noise Models -------- #

def add_gaussian_noise(img):
    mean = 0
    sigma = 25
    gaussian = np.random.normal(mean, sigma, img.shape)
    noisy = img + gaussian
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)

    cv2.imwrite(f"{OUTPUT_DIR}/gaussian_noise.png", noisy)
    return noisy


def add_salt_pepper(img):
    noisy = img.copy()
    prob = 0.02

    rand = np.random.rand(*img.shape)
    noisy[rand < prob/2] = 0
    noisy[rand > 1 - prob/2] = 255

    cv2.imwrite(f"{OUTPUT_DIR}/salt_pepper_noise.png", noisy)
    return noisy


# -------- Restoration Filters -------- #

def apply_filters(noisy, prefix):
    mean = cv2.blur(noisy, (5,5))
    median = cv2.medianBlur(noisy, 5)
    gaussian = cv2.GaussianBlur(noisy, (5,5), 0)

    cv2.imwrite(f"{OUTPUT_DIR}/{prefix}_mean.png", mean)
    cv2.imwrite(f"{OUTPUT_DIR}/{prefix}_median.png", median)
    cv2.imwrite(f"{OUTPUT_DIR}/{prefix}_gaussian.png", gaussian)

    return mean, median, gaussian


# -------- Metrics -------- #

def mse(original, restored):
    return np.mean((original - restored) ** 2)


def psnr(original, restored):
    error = mse(original, restored)
    if error == 0:
        return 100
    return 20 * math.log10(255 / math.sqrt(error))


def evaluate(original, restored_set, label):
    names = ["Mean", "Median", "Gaussian"]
    results = {}

    print(f"\n--- Performance for {label} Noise ---")

    for i in range(3):
        m = mse(original, restored_set[i])
        p = psnr(original, restored_set[i])

        results[names[i]] = (m, p)

        print(f"{names[i]} Filter -> MSE: {m:.2f} | PSNR: {p:.2f}")

    return results


# -------- Analytical Discussion -------- #

def analyze(g_results, sp_results):

    print("\n=========== Analytical Comparison ===========")

    best_g = max(g_results, key=lambda k: g_results[k][1])
    best_sp = max(sp_results, key=lambda k: sp_results[k][1])

    print(f"\nBest filter for Gaussian Noise: {best_g}")
    print("Reason:")
    print("- Gaussian filter matches Gaussian noise distribution")
    print("- Smooths noise while preserving structure")

    print(f"\nBest filter for Salt-Pepper Noise: {best_sp}")
    print("Reason:")
    print("- Median filter removes impulse noise effectively")
    print("- Preserves edges better than averaging")

    print("\nTheory Justification:")
    print("- Mean filter reduces random noise but blurs edges")
    print("- Median filter excellent for impulse noise")
    print("- Gaussian filter ideal for sensor noise modeling")
    print("============================================\n")


# ================= MAIN FLOW ================= #

# Task 1
original = load_image()

# Task 2
gaussian_noisy = add_gaussian_noise(original)
sp_noisy = add_salt_pepper(original)

# Task 3
g_restored = apply_filters(gaussian_noisy, "gaussian")
sp_restored = apply_filters(sp_noisy, "saltpepper")

# Task 4
g_results = evaluate(original, g_restored, "Gaussian")
sp_results = evaluate(original, sp_restored, "Salt-Pepper")

# Task 5
analyze(g_results, sp_results)

print("Processing complete! Check outputs folder.")



 Image Restoration & Noise Modeling System 
 Tasks:
 1. Load surveillance-style image
 2. Add Gaussian & Salt-Pepper noise
 3. Restore using spatial filters
 4. Compute MSE & PSNR
 5. Print analytical comparison



Enter surveillance image path:  corridor.jpg


Original image saved.

--- Performance for Gaussian Noise ---
Mean Filter -> MSE: 39.88 | PSNR: 32.12
Median Filter -> MSE: 46.68 | PSNR: 31.44
Gaussian Filter -> MSE: 48.19 | PSNR: 31.30

--- Performance for Salt-Pepper Noise ---
Mean Filter -> MSE: 31.88 | PSNR: 33.10
Median Filter -> MSE: 12.38 | PSNR: 37.20
Gaussian Filter -> MSE: 26.53 | PSNR: 33.89

=========== Analytical Comparison ===========

Best filter for Gaussian Noise: Mean
Reason:
- Gaussian filter matches Gaussian noise distribution
- Smooths noise while preserving structure

Best filter for Salt-Pepper Noise: Median
Reason:
- Median filter removes impulse noise effectively
- Preserves edges better than averaging

Theory Justification:
- Mean filter reduces random noise but blurs edges
- Median filter excellent for impulse noise
- Gaussian filter ideal for sensor noise modeling

Processing complete! Check outputs folder.
